<a href="https://colab.research.google.com/github/ankitta-singh/machinelearning/blob/main/06_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving house-prices-advanced-regression-techniques.zip to house-prices-advanced-regression-techniques.zip


In [2]:
import zipfile

with zipfile.ZipFile("house-prices-advanced-regression-techniques.zip", "r") as zip_ref:
    zip_ref.extractall("house_data")

In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from xgboost import XGBRegressor

In [17]:
df = pd.read_csv("house_data/train.csv")

In [18]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [19]:
X = df.drop("SalePrice",axis = 1)
y = df["SalePrice"]

In [20]:
cat_cols =X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns
print("categorical columns:", len(cat_cols))
print("numerical column:", len(num_cols))

categorical columns: 43
numerical column: 37


In [23]:
missing = X.isnull().sum()

missing[missing > 0]

,0
LotFrontage,259
Alley,1369
MasVnrType,872
MasVnrArea,8
BsmtQual,37
BsmtCond,37
BsmtExposure,38
BsmtFinType1,37
BsmtFinType2,38
Electrical,1


In [24]:
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

In [25]:
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

In [26]:
X.isnull().sum().sum()

np.int64(0)

In [28]:
X = pd.get_dummies(X, drop_first=True)
X.shape


(1460, 245)

In [29]:
X.dtypes.value_counts()

,count
bool,208
int64,34
float64,3


In [37]:
X_train, x_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.36,
    random_state=42)

In [41]:
X_val, X_test, y_val, y_test = train_test_split(
    x_temp,
    y_temp,
    test_size=0.5556,
    random_state=42
)

In [42]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (934, 245)
Validation: (233, 245)
Test: (293, 245)


In [43]:
xgb_model = XGBRegressor(
    random_state=42
)

In [44]:
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [45]:
y_val_pred_xgb = xgb_model.predict(X_val)

In [46]:
y_val_pred_xgb[:10]

array([358694.47, 100367.5 , 334912.2 , 132867.12, 157666.02,  98809.22,
       125323.72, 131928.77, 172979.6 , 184265.33], dtype=float32)

In [51]:
print("Basic XGBoost Validation Results")

print("MAE :", mean_absolute_error(y_val, y_val_pred_xgb))

mse = mean_squared_error(y_val, y_val_pred_xgb)
print("MSE :", mse)

print("RMSE:", np.sqrt(mse))

print("R2  :", r2_score(y_val, y_val_pred_xgb))

Basic XGBoost Validation Results
MAE : 17135.14453125
MSE : 556349376.0
RMSE: 23587.05950304107
R2  : 0.9064298868179321


In [52]:
param_grid_xgb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5],
    "min_child_weight": [1, 3],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

In [53]:
from sklearn.model_selection import GridSearchCV

In [54]:
grid_search_xgb = GridSearchCV(
    estimator=XGBRegressor(
        random_state=42
    ),
    param_grid=param_grid_xgb,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

In [55]:
grid_search_xgb.fit(X_train, y_train)

Fitting 5 folds for each of 64 candidates, totalling 320 fits


GridSearchCV(cv=5,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=True, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None,...
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 1.0],
                         'learning_rate': [0.05, 0.1], 'max_depth': [3, 5],
                         'min_child_weight': [1, 3], 'n_estimators': [100, 200],
                         'subsample': [0.8, 1.0]},
             scoring='neg_mean_squared_error', verbose=1)

In [58]:
best_xgb = grid_search_xgb.best_estimator_
best_xgb

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=1, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [59]:
y_val_pred_tuned_xgb = best_xgb.predict(X_val)

In [60]:
y_val_pred_tuned_xgb[:10]

array([433274.7  ,  97544.88 , 272121.22 , 131539.7  , 178849.6  ,
        96770.74 , 119116.625, 127576.6  , 177464.34 , 165430.52 ],
      dtype=float32)

In [61]:
print("Tuned XGBoost Validation Results")

print("MAE :", mean_absolute_error(y_val, y_val_pred_tuned_xgb))

mse = mean_squared_error(y_val, y_val_pred_tuned_xgb)
print("MSE :", mse)

print("RMSE:", np.sqrt(mse))

print("R2  :", r2_score(y_val, y_val_pred_tuned_xgb))

Tuned XGBoost Validation Results
MAE : 14817.060546875
MSE : 461643040.0
RMSE: 21485.8800145584
R2  : 0.9223581552505493


In [62]:
y_test_pred_tuned_xgb = best_xgb.predict(X_test)

In [63]:
print("Tuned XGBoost Test Results")

print("MAE :", mean_absolute_error(y_test, y_test_pred_tuned_xgb))

mse = mean_squared_error(y_test, y_test_pred_tuned_xgb)
print("MSE :", mse)

print("RMSE:", np.sqrt(mse))

print("R2  :", r2_score(y_test, y_test_pred_tuned_xgb))


Tuned XGBoost Test Results
MAE : 16839.072265625
MSE : 1042121280.0
RMSE: 32281.903289614136
R2  : 0.8751844167709351
